# 🎓 AI 是怎么"学会"的?
## 🚀 Google Colab 一键运行版

> **跟着视频,5 分钟亲手训练一个 AI**
>
> 不需要安装任何东西,不需要会写代码,**只需要按 ▶️ 运行键**

---

### 🎯 你将经历什么?

| 步骤 | 你要做的 | 大约耗时 |
|------|---------|---------|
| 1. 准备数据 | 按 ▶️ 运行 | 30 秒 |
| 2. 配置 AI | 按 ▶️ 运行 | 10 秒 |
| 3. 训练 AI | 按 ▶️ 运行 | 2-3 分钟 |
| 4. 测试 AI | 上传一张图,按 ▶️ 运行 | 5 秒 |

**总耗时不到 5 分钟,真正能训练出一个准确率 95%+ 的 AI。**

---

### ⚠️ 开始前请做这一步(只做一次)

**点击顶部菜单 → "代码执行程序" → "更改运行时类型" → 硬件加速器选 `T4 GPU` → 保存**

> 不开 GPU 也能跑,但训练会从 3 分钟变成 30 分钟 😅

---


## 0️⃣ 环境检查

先确认 Colab 已经分配了 GPU,以及 fastai 工具包是否就位。

> 💡 这一步就像新员工入职第一天 —— 先看一下办公电脑能不能开机。


In [ ]:
# 检查是否有 GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU 已就位: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 没检测到 GPU。建议:菜单 → 代码执行程序 → 更改运行时类型 → 选 T4 GPU")

# 检查 fastai 是否可用(Colab 自带)
import fastai
print(f"✅ fastai 版本: {fastai.__version__}")


## 📦 Step 1:准备"教材"——下载猫狗图片

### 🏢 职场类比

```
HR 给新员工准备入职培训:
┌─────────────────────────────────┐
│  📁 培训资料/                    │
│      ├── 📁 产品知识/  (一堆 PPT) │
│      └── 📁 公司制度/  (一堆 PPT) │
└─────────────────────────────────┘
                  ↓ 一模一样的逻辑
┌─────────────────────────────────┐
│  📁 dog_or_not/                 │
│      ├── 📁 cat/    (200 张猫图) │
│      └── 📁 dog/    (200 张狗图) │
└─────────────────────────────────┘
```

**关键认知**:AI 学习的方式和人非常像 —— **看大量例子**。
- 给它看 200 张猫 → 它就知道"猫长这样"
- 给它看 200 张狗 → 它就知道"狗长这样"
- 文件夹名(`cat` / `dog`)就是"答案"

> 下面这段代码会自动下载一个真实的猫狗图片库(fastai 官方提供),
> 然后整理成 cat / dog 两个文件夹。你只需要按 ▶️ 。


In [ ]:
# 下载 fastai 官方的宠物图片数据集(只下载一次,会缓存)
from fastai.vision.all import *
import shutil, random

print("📥 正在下载图片库(首次约 30 秒)...")
pets_path = untar_data(URLs.PETS) / 'images'

# 准备 cat / dog 两个文件夹
target = Path('dog_or_not')
if target.exists():
    shutil.rmtree(target)
(target / 'cat').mkdir(parents=True)
(target / 'dog').mkdir(parents=True)

# 在这个数据集里,猫的文件名以大写字母开头,狗以小写字母开头
random.seed(42)
all_imgs = [f for f in pets_path.ls() if f.suffix.lower() in ['.jpg', '.jpeg']]
cats = [f for f in all_imgs if f.name[0].isupper()]
dogs = [f for f in all_imgs if not f.name[0].isupper()]

# 每类取 200 张,加快训练速度
for img in random.sample(cats, 200):
    shutil.copy(img, target / 'cat' / img.name)
for img in random.sample(dogs, 200):
    shutil.copy(img, target / 'dog' / img.name)

print(f"✅ 教材准备完毕")
print(f"   🐱 猫:{len(list((target/'cat').ls()))} 张")
print(f"   🐕 狗:{len(list((target/'dog').ls()))} 张")


## 🧠 Step 2:找一个"现成的大脑"——迁移学习

### 🏢 职场类比

```
❌ 招应届生从 0 开始教 → 要 6 个月才能独立工作
       (从零训练 AI → 要几百万张图、几周时间、几万元成本)

✅ 招有 3 年经验的候选人 → 已经会通用技能,只需培训公司业务,1 周上手
       (用预训练模型 → 几分钟微调搞定)
```

**ResNet18 就是那个"3 年经验的候选人"**:
- 它已经被 100 多万张图片训练过
- 它认识边缘、颜色、形状、纹理这些"通用视觉技能"
- 我们只需要再教它一件小事:**区分猫和狗**

这个过程叫 **迁移学习 (Transfer Learning)**。

> 💡 全世界几十万个预训练模型都在 **Hugging Face** 免费下载。
> 这就是 AI 发展这么快的核心原因:**所有人都可以站在巨人的肩膀上**。

---

### 下面这段代码做了什么?

```
告诉 AI:
  1. 输入是图片,输出是分类(猫 or 狗)
  2. 自动找出文件夹里所有图片
  3. 80% 用来"培训"、20% 留着"考核"
  4. 文件夹名就是答案
  5. 所有图片缩到 192×192 的统一尺寸
```


In [ ]:
# 配置"数据加载器"——告诉 AI 怎么读这些图片
path = Path('dog_or_not')

dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),               # 输入=图片, 输出=分类
    get_items=get_image_files,                        # 自动找所有图片
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # 80% 培训 / 20% 考核
    get_y=parent_label,                               # 用父文件夹名当答案
    item_tfms=[Resize(192, method='squish')]          # 统一缩放 192×192
).dataloaders(path, bs=32)

# 预览一下教材长啥样
dls.show_batch(max_n=6, figsize=(8, 6))


## 🏋️ Step 3:开始训练——核心只有 2 行代码

### 🏢 职场类比

```
新员工入职后的培训流程:

第 1 轮培训 →  看一遍所有资料,搞错 30% 的题
第 2 轮培训 →  又看一遍,错 15%
第 3 轮培训 →  再看一遍,错 2%

✅ 培训完成,可以独立工作了
```

**AI 训练一模一样**:
- 把所有图片反复看 3 遍(叫 3 个 epoch)
- 每看一遍,都在调整"自己的判断方式"
- **错误率从 30% → 15% → 2% 肉眼可见地下降**

> 🎬 **运行下面这格,准备好看错误率下降的画面 —— 这是 AI 在"思考"。**


In [ ]:
# 创建一个"学习器":数据 + 大脑 + 评估方法
learn = vision_learner(dls, resnet18, metrics=error_rate)

# 开始微调,反复学 3 轮(GPU 下约 2-3 分钟)
learn.fine_tune(3)


### 📊 怎么看这个训练表?

| 列名 | 你需要懂的 |
|------|----------|
| `epoch` | 第几轮培训 |
| `train_loss` | 培训时的犯错程度(越低越好) |
| `valid_loss` | 考核时的犯错程度(越低越好) |
| `error_rate` | 错误率 ← **这是最重要的指标,越低越好** |

**判断 AI 学得好不好:**
- ✅ `error_rate` 一直下降 → AI 真的在学
- ❌ `error_rate` 不动 → 数据或方法有问题
- ⚠️ `train_loss` 很低但 `valid_loss` 很高 → AI 死记硬背了(叫**过拟合**)

> 💡 **过拟合的职场类比**:新员工把培训手册背得滚瓜烂熟,
> 但一遇到客户的实际问题就傻眼 —— 这种员工你也不想要。


## 🎯 Step 4:考试——AI 真的学会了吗?

### 🏢 职场类比

```
新员工培训完,试用期考核:
  → 拿没培训过的真实案例给他做
  → 看他能不能处理对
  → 这就是 AI 训练里的"验证集"
```

下面看一下 AI 在它没见过的图片上表现如何。


In [ ]:
# AI 在验证集(它没见过的图)上的表现
learn.show_results(max_n=6, figsize=(10, 8))


## 📷 Step 4.5:用你自己的图测试一下!

### 🏢 职场类比

```
试用期通过后,给他真实工作任务:
  → 完全没见过的客户、完全没见过的问题
  → 这才是真正的能力检验
```

**运行下一格,会弹出"选择文件"按钮 —— 上传一张你自己的猫或狗的照片。**

> 💡 也可以试试模糊图、动漫猫、毛绒玩具,看 AI 怎么判断。


In [ ]:
# 上传你自己的图片
from google.colab import files
uploaded = files.upload()


In [ ]:
# 用 AI 判断你刚才上传的图片
img_path = list(uploaded.keys())[0]
img = PILImage.create(img_path)

# AI 给出判断
result, _, probs = learn.predict(img)

# 解析结果
cat_idx = learn.dls.vocab.o2i['cat']
dog_idx = learn.dls.vocab.o2i['dog']

# 展示图片 + 结果
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f"👉 AI 的判断: {'狗 🐕' if result == 'dog' else '猫 🐱'}", fontsize=16)
plt.show()

print(f"🐱 猫的概率: {probs[cat_idx]*100:.2f}%")
print(f"🐕 狗的概率: {probs[dog_idx]*100:.2f}%")


### ⚠️ 重要认知:AI 给的是"概率",不是"答案"

```
错误的理解:                  正确的理解:
   AI 说"这是狗"  →  ✅      AI 说"99.98% 是狗"  →  ✅
   所以一定是狗                所以很可能是狗,但有 0.02% 是错的
```

**这就是为什么 ChatGPT 也会出错** —— 它本质上和这个猫狗分类器一样,
在挑"**最可能对的答案**"告诉你,而不是"**绝对正确的答案**"。

> 💡 这条认知用在职场上最关键:
> - 用 AI 筛简历 → 别完全信结果,要看"匹配度评分"
> - 用 AI 做数据分析 → 重要数字一定自己核对
> - 用 AI 写报告 → 结构交给它,关键事实自己审


## 💾 Step 5:保存"大脑"——一个文件可以无限复制

### 🏢 职场类比

```
传统培训:
  ❌ 每招一个新员工,都得从头培训一遍
  ❌ 培训成本随员工数量线性增长

AI 模型:
  ✅ 训练一次,导出成一个文件(约 50MB)
  ✅ 发给任何人,一行代码就能加载
  ✅ 100 个人用,成本和 1 个人用一样
```

**这就是为什么 AI 能颠覆所有行业**:
> 高水平的认知能力,第一次很贵,后面**无限复制成本接近 0**。


In [ ]:
# 把训练好的 AI 保存成一个文件
learn.export('dog_cat_model.pkl')

# 下载到本地电脑
from google.colab import files
files.download('dog_cat_model.pkl')

print("✅ AI 大脑文件已开始下载")
print("📦 这个 .pkl 文件就是这个 AI 的完整大脑")
print("📤 你可以发给任何人,他们用一行代码就能加载使用")


### 🔁 别人怎么用这个 .pkl 文件?

只需要 3 行代码:

```python
from fastai.vision.all import *

# 加载训练好的大脑
learn = load_learner('dog_cat_model.pkl')

# 直接预测
result = learn.predict(PILImage.create('某张图.jpg'))
```

**就这么简单。** 这就是 AI 时代认知能力的可复制性。


---

## 🎬 总结:你刚才做了什么?

### 1️⃣ AI 的"学习"=看大量例子 + 反复调整
和教 3 岁小孩、培训新员工的逻辑一模一样。

### 2️⃣ 现代 AI 都站在巨人的肩膀上
没人从零开始训练,大家都在用 Hugging Face 上的预训练模型微调。

### 3️⃣ AI 输出的是"概率",不是"绝对答案"
这就是它会"幻觉"的根本原因。**重要决策必须人工核对**。

### 4️⃣ 一个模型可以无限复制
训练贵,使用便宜。这是 AI 经济学的核心。

---

## 💼 今天的认知怎么用在工作里?

| 场景 | 错误做法 | 正确做法 |
|------|---------|---------|
| HR 用 AI 筛简历 | ❌ 让 AI 决定录用谁 | ✅ 让 AI 输出"匹配度评分",最终决定权在你 |
| 用 AI 写报告 | ❌ 让 AI 生成销售数据 | ✅ 结构交给 AI,数字自己填 |
| 用 AI 出图 | ❌ "穿西装的男人" | ✅ "30 岁中国男性、深蓝色西装、办公室背景、正在打电话" |
| 用 AI 做客户分析 | ❌ 直接问"我的客户怎么样" | ✅ 把历史数据整理好喂给它,让它找规律 |

---

## 📺 下一课预告

**第 6 课:怎么让 AI 学会你公司的内部资料?**

→ 这一课你训练的是"通用大脑"
→ 下一课带你做"专属大脑"——只懂你公司业务、只为你服务的 AI

---

> ### 🙌 觉得有帮助?
> - **点赞 + 收藏 + 关注**
> - 评论区告诉我:你最希望 AI 帮你解决工作里的什么问题?
> - 下期见 👋
